In [ ]:
"""
Alpha Trading Research Platform

Notebook:
Sprint08b_FTSE100_Universe

Purpose:
Show how to swap the universe to a different set of tickers -
FTSE 100 as the example - without editing any code, and flag the
things that are genuinely different about a bigger, non-US universe.

Author:
Alison

Version:
0.10
"""

In [ ]:
from alpha.config import Config
from alpha.universes import load_universe
from alpha.data import get_prices, get_monthly_prices, get_benchmark_prices
from alpha.regime import calculate_regime
from alpha.scanner import scan_latest, top_opportunities

In [ ]:
# =====================================================
# LOAD THE UNIVERSE FROM FILE
# =====================================================
# universes/ftse100.csv is a STARTING POINT, not a verified live list -
# see the comment header in that file. Cross-check it against an
# authoritative source (e.g. londonstockexchange.com) before trusting
# it, and edit the CSV directly once you have a verified list - no
# code changes needed here.

ftse_tickers = load_universe("../universes/ftse100.csv")
print(f"Loaded {len(ftse_tickers)} tickers")
print(ftse_tickers[:5], "...")

In [ ]:
# =====================================================
# CONFIG FOR THE LARGER UNIVERSE
# =====================================================
# Two changes worth making deliberately, not just plugging in the new
# ticker list:
#
# 1. top_stocks - with ~90 names instead of 10, holding only the top 3
#    is a much smaller slice of the universe. Bumped up here; there's
#    no single right number, it's a judgement call about how
#    concentrated vs. diversified you want the portfolio to be.
#
# 2. regime_benchmark - SPY tracks the S&P 500, not relevant to UK
#    stocks. Switched to "^FTSE", the FTSE 100 index itself.

config = Config(
    universe=ftse_tickers,
    top_stocks=10,
    regime_benchmark="^FTSE",
)
config

In [ ]:
# =====================================================
# DATA
# =====================================================
# ~90 tickers takes noticeably longer to download than 10 - expect
# this cell to take a while longer than the same step in earlier
# notebooks.

print("Downloading FTSE 100 market data...")

prices = get_prices(config)
monthly_prices = get_monthly_prices(prices)

print("Downloading FTSE 100 index for regime filter...")

benchmark_prices = get_benchmark_prices(config)
monthly_benchmark = benchmark_prices.resample("ME").last()
regime = calculate_regime(monthly_benchmark, config)

In [ ]:
# =====================================================
# SCAN
# =====================================================

scan = scan_latest(monthly_prices, regime, config)

if scan.empty:
    print("No opportunities flagged this month.")
else:
    display(scan)
    print()
    display(top_opportunities(scan, n=5))

In [ ]:
# =====================================================
# NOTES
# =====================================================
# One thing worth understanding, not just a code detail: mixing
# currencies. yfinance returns FTSE stock prices in GBX/GBP, not
# converted to any common currency. Every calculation in this package
# (momentum, returns, backtests) works on PERCENTAGE changes, so the
# math itself is fine regardless of currency - a UK stock's % return
# and a US stock's % return combine correctly even though their
# absolute prices are in different currencies.
#
# What's NOT accounted for: currency risk. If you ever combine US and
# UK tickers in the SAME universe, your USD-denominated view of a GBP
# stock's return would also move with the GBP/USD exchange rate,
# which none of this package's math currently adjusts for. Running
# FTSE 100 on its own (as here) sidesteps this entirely - it's only a
# consideration if you deliberately mix markets in one universe.
#
# Also worth knowing: the dashboard's sidebar doesn't have a universe
# switcher yet - that's Config.universe still, and today it's still
# hardcoded to the original 10-ticker list at the top of dashboard.py.
# If you want the dashboard itself to run against FTSE 100, that's a
# small follow-up change, not something this notebook does for you.